In [1]:
import pandas as pd

# 1. Load Dataset
file_path = "data/student_exam_performance.csv"
df = pd.read_csv(file_path)

print("=" * 60)
print("INFORMASI AWAL DATASET")
print("=" * 60)

# 2. Cek Jumlah Baris dan Kolom
jumlah_baris, jumlah_kolom = df.shape
print(f"Jumlah Baris : {jumlah_baris:,}")
print(f"Jumlah Kolom : {jumlah_kolom}")
print("-" * 60)

# 3. Cek Missing Value (Nilai Kosong / NaN)
missing_values = df.isnull().sum()
total_missing = missing_values.sum()
print("INFORMASI MISSING VALUE:")
if total_missing == 0:
    print("Tidak ditemukan missing value pada dataset.")
else:
    print(f"Total missing values: {total_missing}")
    kolom_missing = missing_values[missing_values > 0]
    for col, val in kolom_missing.items():
        persen = (val / jumlah_baris) * 100
        print(f"- {col}: {val} ({persen:.2f}%)")
print("-" * 60)

# 4. Cek Data Duplikat
jumlah_duplikat = df.duplicated().sum()
print("INFORMASI DATA DUPLIKAT:")
if jumlah_duplikat == 0:
    print("Tidak ada baris duplikat.")
else:
    print(f"Ditemukan {jumlah_duplikat} baris duplikat.")
print("-" * 60)

# 5. Cek Distribusi Kelas Target
# Kolom target potensial dalam dataset: 'pass_status', 'performance_grade', 'performance_level'
target_columns = ["pass_status", "performance_grade", "performance_level"]

print("DISTRIBUSI KELAS TARGET:")
for target in target_columns:
    if target in df.columns:
        print(f"\n--- Distribusi Target: '{target}' ---")
        counts = df[target].value_counts()
        percentages = df[target].value_counts(normalize=True) * 100
        summary_target = pd.DataFrame({"Jumlah": counts, "Persentase (%)": percentages.round(2)})
        print(summary_target)
print("=" * 60)


INFORMASI AWAL DATASET
Jumlah Baris : 100,000
Jumlah Kolom : 44
------------------------------------------------------------
INFORMASI MISSING VALUE:
Total missing values: 52161
- parent_education: 6526 (6.53%)
- previous_gpa: 7793 (7.79%)
- attendance_percentage: 9863 (9.86%)
- notes_quality: 8407 (8.41%)
- sleep_quality: 6994 (6.99%)
- device_availability: 2965 (2.96%)
- time_management_score: 9613 (9.61%)
------------------------------------------------------------


INFORMASI DATA DUPLIKAT:
Tidak ada baris duplikat.
------------------------------------------------------------
DISTRIBUSI KELAS TARGET:

--- Distribusi Target: 'pass_status' ---
             Jumlah  Persentase (%)
pass_status                        
Pass          77389           77.39
Fail          22611           22.61

--- Distribusi Target: 'performance_grade' ---
                   Jumlah  Persentase (%)
performance_grade                        
D                   34886           34.89
F                   22611           22.61
C                   21794           21.79
B                   13190           13.19
A                    7519            7.52

--- Distribusi Target: 'performance_level' ---
                   Jumlah  Persentase (%)
performance_level                        
Low                 45123           45.12
Medium              41999           42.00
High                12878           12.88


In [2]:

import pandas as pd

# ==============================================================================
# 1. LOAD DATASET
# ==============================================================================
file_path = "data/student_exam_performance.csv"
df = pd.read_csv(file_path)

# ==============================================================================
# 2. DEFINISI STRUKTUR KOLOM
# ==============================================================================

# a) 13 Fitur Prediktor Terpilih (X) — Fokus kelompok dan nama fitur saja
FITUR_13 = [
    {"Kelompok": "Riwayat akademik", "Fitur": "previous_exam_score"},
    {"Kelompok": "Riwayat akademik", "Fitur": "previous_gpa"},
    {"Kelompok": "Riwayat akademik", "Fitur": "attendance_percentage"},
    {"Kelompok": "Riwayat akademik", "Fitur": "assignment_completion_rate"},
    {"Kelompok": "Kebiasaan belajar", "Fitur": "study_hours_per_day"},
    {"Kelompok": "Kebiasaan belajar", "Fitur": "self_study_hours"},
    {"Kelompok": "Kebiasaan belajar", "Fitur": "practice_tests_completed"},
    {"Kelompok": "Kebiasaan belajar", "Fitur": "exam_preparation_days"},
    {"Kelompok": "Gaya hidup / mental", "Fitur": "sleep_hours"},
    {"Kelompok": "Gaya hidup / mental", "Fitur": "stress_level"},
    {"Kelompok": "Gaya hidup / mental", "Fitur": "exam_anxiety_level"},
    {"Kelompok": "Kondisi ujian", "Fitur": "exam_difficulty"},
    {"Kelompok": "Latar belakang", "Fitur": "family_income"},
]

# b) 1 Kolom Target (y)
TARGET_Y = ["pass_status"]

# c) 5 Kolom Dibuang karena Data Leakage / ID (Tanpa Alasan)
KOLOM_LEAKAGE_ID = [
    {"Kolom": "exam_score", "Kategori": "Data Leakage"},
    {"Kolom": "performance_grade", "Kategori": "Data Leakage"},
    {"Kolom": "performance_level", "Kategori": "Data Leakage"},
    {"Kolom": "questions_correct", "Kategori": "Data Leakage"},
    {"Kolom": "student_id", "Kategori": "Pengenal (ID)"},
]

# d) 25 Kolom Dibuang karena Tidak Relevan (Tanpa Alasan)
LIST_TIDAK_RELEVAN = [
    "age", "gender", "education_level", "school_type", "parent_education", "urban_rural",
    "class_participation", "study_consistency", "study_environment", "study_method",
    "revision_frequency", "notes_quality", "private_tuition", "time_management_score",
    "sleep_quality", "daily_screen_time", "physical_activity_hours", "break_frequency",
    "motivation_level", "internet_access", "device_availability",
    "educational_app_usage", "online_learning_hours", "online_course_hours",
    "questions_attempted"
]

KOLOM_TIDAK_RELEVAN = [
    {"Kolom": col, "Kategori": "Tidak Relevan"}
    for col in LIST_TIDAK_RELEVAN
]

# e) Pengelompokan seluruh 44 kolom dataset awal berdasarkan domain
PENGELOMPOKAN_SEMUA_KOLOM = {
    "Identitas": ["student_id"],
    "Demografi & Latar Belakang": ["age", "gender", "education_level", "school_type", "family_income", "parent_education", "urban_rural"],
    "Riwayat Akademik": ["previous_exam_score", "previous_gpa", "attendance_percentage", "assignment_completion_rate", "class_participation"],
    "Kebiasaan & Metode Belajar": ["study_hours_per_day", "self_study_hours", "private_tuition", "study_consistency", "study_environment", "study_method", "revision_frequency", "notes_quality", "practice_tests_completed", "exam_preparation_days"],
    "Pemanfaatan Teknologi / Online": ["online_learning_hours", "internet_access", "device_availability", "educational_app_usage", "online_course_hours"],
    "Gaya Hidup & Kondisi Mental": ["sleep_hours", "sleep_quality", "daily_screen_time", "physical_activity_hours", "break_frequency", "stress_level", "motivation_level", "time_management_score", "exam_anxiety_level"],
    "Karakteristik & Hasil Ujian": ["exam_difficulty", "questions_attempted", "questions_correct", "exam_score", "performance_grade", "pass_status", "performance_level"]
}

# ==============================================================================
# 3. PENGECEKAN VALIDASI (ASSERT)
# ==============================================================================
list_13_fitur = [item["Fitur"] for item in FITUR_13]
list_leakage = [item["Kolom"] for item in KOLOM_LEAKAGE_ID]
list_tidak_relevan = [item["Kolom"] for item in KOLOM_TIDAK_RELEVAN]

total_terklasifikasi = list_13_fitur + TARGET_Y + list_leakage + list_tidak_relevan

# Memastikan keunikan dan kesesuaian jumlah kolom (44 kolom)
assert len(total_terklasifikasi) == len(set(total_terklasifikasi)), "Ada kolom yang terdaftar ganda!"
assert len(total_terklasifikasi) == 44, f"Total kolom harus 44, ditemukan {len(total_terklasifikasi)}"
assert set(total_terklasifikasi) == set(df.columns), "Kolom yang didefinisikan tidak sesuai dengan dataset CSV!"

# ==============================================================================
# 4. TAMPILAN OUTPUT
# ==============================================================================

# ------------------------------------------------------------------------------
# 1. SEMUA FITUR AWAL (44 KOLOM DIKELOMPOKKAN)
# ------------------------------------------------------------------------------
print("=" * 80)
print("1. DAFTAR SEMUA FITUR AWAL DATASET (44 KOLOM DIKELOMPOKKAN)")
print("=" * 80)

daftar_semua = []
for kelompok, cols in PENGELOMPOKAN_SEMUA_KOLOM.items():
    for col in cols:
        daftar_semua.append({
            "Kelompok Domain": kelompok,
            "Nama Kolom": col,
            "Tipe Data": str(df[col].dtype),
            "Contoh Nilai": str(df[col].dropna().iloc[0])
        })

df_semua_kolom = pd.DataFrame(daftar_semua)
print(df_semua_kolom.to_string(index=False))

# ------------------------------------------------------------------------------
# 2. 13 FITUR PILIHAN (X)
# ------------------------------------------------------------------------------
print("\n" + "=" * 80)
print("2. FITUR PREDIKTOR TERPILIH (13 FITUR X)")
print("=" * 80)
df_fitur_13 = pd.DataFrame(FITUR_13)
print(df_fitur_13.to_string(index=False))

# ------------------------------------------------------------------------------
# 3. TARGET (Y) DAN DISTRIBUSI KELASNYA
# ------------------------------------------------------------------------------
print("\n" + "=" * 80)
print("3. TARGET PREDIKSI (Y): pass_status")
print("=" * 80)
target_nama = TARGET_Y[0]
counts = df[target_nama].value_counts()
percentages = df[target_nama].value_counts(normalize=True) * 100

tabel_target = pd.DataFrame({
    "Kelas": counts.index,
    "Jumlah Baris": counts.values,
    "Persentase (%)": percentages.round(2).values
})
print(tabel_target.to_string(index=False))

# ------------------------------------------------------------------------------
# 4. KOLOM YANG DIBUANG (HANYA KOLOM DAN KATEGORI)
# ------------------------------------------------------------------------------
print("\n" + "=" * 80)
print("4. DAFTAR KOLOM YANG DIBUANG (30 KOLOM: 5 LEAKAGE/ID + 25 TIDAK RELEVAN)")
print("=" * 80)
df_dibuang = pd.DataFrame(KOLOM_LEAKAGE_ID + KOLOM_TIDAK_RELEVAN)
print(df_dibuang.to_string(index=False))

# ------------------------------------------------------------------------------
# 5. RINGKASAN AKHIR PERANAN KOLOM
# ------------------------------------------------------------------------------
print("\n" + "=" * 80)
print("RINGKASAN PERANAN SELURUH KOLOM")
print("=" * 80)
df_ringkasan = pd.DataFrame({
    "Peranan": [
        "Fitur Terpilih (X)",
        "Target Prediksi (y)",
        "Dibuang (Data Leakage & ID)",
        "Dibuang (Tidak Relevan)",
        "Total Kolom Dataset"
    ],
    "Jumlah Kolom": [
        len(list_13_fitur),
        len(TARGET_Y),
        len(list_leakage),
        len(list_tidak_relevan),
        len(total_terklasifikasi)
    ]
})
print(df_ringkasan.to_string(index=False))
print("=" * 80)

1. DAFTAR SEMUA FITUR AWAL DATASET (44 KOLOM DIKELOMPOKKAN)
               Kelompok Domain                 Nama Kolom Tipe Data Contoh Nilai
                     Identitas                 student_id    object   STU_000001
    Demografi & Latar Belakang                        age     int64           20
    Demografi & Latar Belakang                     gender    object       Female
    Demografi & Latar Belakang            education_level    object  High School
    Demografi & Latar Belakang                school_type    object       Public
    Demografi & Latar Belakang              family_income    object       Middle
    Demografi & Latar Belakang           parent_education    object       Master
    Demografi & Latar Belakang                urban_rural    object        Rural
              Riwayat Akademik        previous_exam_score   float64        78.36
              Riwayat Akademik               previous_gpa   float64         2.95
              Riwayat Akademik      attendance_pe

In [3]:
import pandas as pd

df = pd.read_csv("data/student_exam_performance.csv")

fitur = [
    "previous_exam_score", "previous_gpa", "attendance_percentage",
    "assignment_completion_rate", "study_hours_per_day", "self_study_hours",
    "practice_tests_completed", "exam_preparation_days", "sleep_hours",
    "stress_level", "exam_anxiety_level", "exam_difficulty", "family_income",
]
target = "pass_status"

data = df[fitur + [target]].copy()   # hanya 13 fitur + 1 target

assert data.shape[1] == 14           # cek: 13 + 1
data.to_csv("data/data_dipilih.csv", index=False)   # opsional, CSV baru
data.head()

,previous_exam_score,previous_gpa,attendance_percentage,assignment_completion_rate,study_hours_per_day,self_study_hours,practice_tests_completed,exam_preparation_days,sleep_hours,stress_level,exam_anxiety_level,exam_difficulty,family_income,pass_status
0,78.36,2.95,81.49,65.18,4.94,2.63,8,27,6.31,10,9.39,Medium,Middle,Pass
1,73.40,2.89,NaN,76.53,3.42,2.64,10,4,6.81,8,6.31,Easy,Middle,Pass
2,82.76,3.35,100.00,70.39,1.06,0.83,5,28,6.53,9,9.87,Easy,Middle,Pass
3,60.61,2.57,84.91,75.06,7.89,5.51,8,16,6.60,10,9.68,Medium,Middle,Pass
4,74.79,2.84,88.21,61.54,2.63,2.29,1,28,7.83,10,10.00,Medium,High,Pass


# ==============================================================================
# JENIS DATA BERDASARKAN SKALA PENGUKURAN
# ==============================================================================

In [4]:
# ------------------------------------------------------------------------------
# 1. Definisi Skala Pengukuran (14 Kolom), Tabel, Rekap, dan Validasi Assert
# ------------------------------------------------------------------------------

# Dictionary pemetaan 14 kolom ke jenis skala pengukurannya
skala_ukur = {
    # Nominal: Kategori murni tanpa urutan nilai intrinsik
    "pass_status": "Nominal",
    
    # Ordinal: Kategori bertingkat / memiliki urutan logis
    "exam_difficulty": "Ordinal",
    "family_income": "Ordinal",
    
    # Interval: Data numerik tanpa nilai nol mutlak
    "previous_exam_score": "Interval",
    "previous_gpa": "Interval",
    "stress_level": "Interval",
    "exam_anxiety_level": "Interval",
    
    # Rasio: Data numerik dengan titik nol mutlak
    "attendance_percentage": "Rasio",
    "assignment_completion_rate": "Rasio",
    "study_hours_per_day": "Rasio",
    "self_study_hours": "Rasio",
    "practice_tests_completed": "Rasio",
    "exam_preparation_days": "Rasio",
    "sleep_hours": "Rasio"
}

# Assert bahwa 14 kolom dalam dictionary sama persis dengan kolom variabel data
assert len(skala_ukur) == 14, f"Jumlah kolom harus 14, tetapi ditemukan {len(skala_ukur)}"
assert set(skala_ukur.keys()) == set(data.columns), "Kolom skala_ukur tidak sesuai dengan kolom variabel data!"

# 1a. Membuat tabel skala pengukuran
df_skala = pd.DataFrame(list(skala_ukur.items()), columns=["Kolom", "Skala"])
print("=== TABEL SKALA PENGUKURAN (14 KOLOM) ===")
print(df_skala.to_string(index=False))

# 1b. Rekapitulasi jumlah kolom per skala pengukuran
rekap_skala = df_skala["Skala"].value_counts().reset_index()
rekap_skala.columns = ["Skala", "Jumlah Kolom"]
print("\n=== REKAPITULASI JUMLAH KOLOM PER SKALA ===")
print(rekap_skala.to_string(index=False))


=== TABEL SKALA PENGUKURAN (14 KOLOM) ===
                     Kolom    Skala
               pass_status  Nominal
           exam_difficulty  Ordinal
             family_income  Ordinal
       previous_exam_score Interval
              previous_gpa Interval
              stress_level Interval
        exam_anxiety_level Interval
     attendance_percentage    Rasio
assignment_completion_rate    Rasio
       study_hours_per_day    Rasio
          self_study_hours    Rasio
  practice_tests_completed    Rasio
     exam_preparation_days    Rasio
               sleep_hours    Rasio

=== REKAPITULASI JUMLAH KOLOM PER SKALA ===
   Skala  Jumlah Kolom
   Rasio             7
Interval             4
 Ordinal             2
 Nominal             1


In [5]:
# ------------------------------------------------------------------------------
# 2. Tabel Nilai Minimum dan Maksimum untuk Kolom Interval dan Rasio
# ------------------------------------------------------------------------------

# Filter kolom numerik ber-skala Interval dan Rasio
kolom_interval_rasio = [col for col, skala in skala_ukur.items() if skala in ["Interval", "Rasio"]]

# Membentuk tabel ringkasan nilai minimum dan maksimum
tabel_min_max = pd.DataFrame({
    "Kolom": kolom_interval_rasio,
    "Skala": [skala_ukur[col] for col in kolom_interval_rasio],
    "Nilai Min": [data[col].min() for col in kolom_interval_rasio],
    "Nilai Max": [data[col].max() for col in kolom_interval_rasio]
})

print("=== TABEL NILAI MINIMUM DAN MAKSIMUM (INTERVAL & RASIO) ===")
print(tabel_min_max.to_string(index=False))


=== TABEL NILAI MINIMUM DAN MAKSIMUM (INTERVAL & RASIO) ===
                     Kolom    Skala  Nilai Min  Nilai Max
       previous_exam_score Interval      30.00      100.0
              previous_gpa Interval       1.00        4.0
              stress_level Interval       5.00       10.0
        exam_anxiety_level Interval       1.00       10.0
     attendance_percentage    Rasio      46.34      100.0
assignment_completion_rate    Rasio      20.00      100.0
       study_hours_per_day    Rasio       0.50       12.0
          self_study_hours    Rasio       0.25       10.0
  practice_tests_completed    Rasio       0.00       20.0
     exam_preparation_days    Rasio       1.00       30.0
               sleep_hours    Rasio       3.00       11.0


In [6]:
# ------------------------------------------------------------------------------
# 3. Kategori Unik Kolom Ordinal & Distribusi Kelas Target Nominal
# ------------------------------------------------------------------------------

# 3a. Menampilkan nilai/kategori unik dari fitur ordinal
print("=== KATEGORI UNIK FITUR ORDINAL ===")
print(f"exam_difficulty : {data['exam_difficulty'].unique().tolist()}")
print(f"family_income   : {data['family_income'].unique().tolist()}")

# 3b. Menampilkan jumlah dan persentase kelas target pass_status (Nominal)
counts_pass = data["pass_status"].value_counts()
pct_pass = data["pass_status"].value_counts(normalize=True) * 100

tabel_pass_status = pd.DataFrame({
    "Kelas": counts_pass.index,
    "Jumlah": counts_pass.values,
    "Persentase (%)": pct_pass.round(2).values
})

print("\n=== DISTRIBUSI KELAS TARGET (pass_status) ===")
print(tabel_pass_status.to_string(index=False))


=== KATEGORI UNIK FITUR ORDINAL ===
exam_difficulty : ['Medium', 'Easy', 'Hard']
family_income   : ['Middle', 'High', 'Lower-Middle', 'Upper-Middle', 'Low']

=== DISTRIBUSI KELAS TARGET (pass_status) ===
Kelas  Jumlah  Persentase (%)
 Pass   77389           77.39
 Fail   22611           22.61


In [7]:
# ------------------------------------------------------------------------------
# 4. Ordinal Encoding untuk exam_difficulty dan family_income
# ------------------------------------------------------------------------------

# Menyimpan salinan data sebelum pengubahan nilai
data_sebelum = data.copy()

# Definisi aturan pemetaan urutan ordinal ke bilangan bulat
mapping_exam_difficulty = {"Easy": 1, "Medium": 2, "Hard": 3}
mapping_family_income = {
    "Low": 1,
    "Lower-Middle": 2,
    "Middle": 3,
    "Upper-Middle": 4,
    "High": 5
}

# Menerapkan ordinal encoding pada kedua kolom
data["exam_difficulty"] = data["exam_difficulty".strip()].map(mapping_exam_difficulty)
data["family_income"] = data["family_income"].map(mapping_family_income)

# Assert untuk memastikan tidak timbul nilai NaN setelah encoding
assert data["exam_difficulty"].isna().sum() == 0, "Ditemukan NaN pada exam_difficulty setelah encoding!"
assert data["family_income"].isna().sum() == 0, "Ditemukan NaN pada family_income setelah encoding!"

print("=== ORDINAL ENCODING BERHASIL ===")
print("Pemetaan exam_difficulty :", mapping_exam_difficulty)
print("Pemetaan family_income   :", mapping_family_income)
print(f"Jumlah NaN exam_difficulty : {data['exam_difficulty'].isna().sum()} (Valid)")
print(f"Jumlah NaN family_income   : {data['family_income'].isna().sum()} (Valid)")


=== ORDINAL ENCODING BERHASIL ===
Pemetaan exam_difficulty : {'Easy': 1, 'Medium': 2, 'Hard': 3}
Pemetaan family_income   : {'Low': 1, 'Lower-Middle': 2, 'Middle': 3, 'Upper-Middle': 4, 'High': 5}
Jumlah NaN exam_difficulty : 0 (Valid)
Jumlah NaN family_income   : 0 (Valid)


In [8]:
# ------------------------------------------------------------------------------
# 5. Perbandingan Nilai Sebelum dan Sesudah untuk Siswa Indeks 56663
# ------------------------------------------------------------------------------

# Mengambil data siswa dengan indeks 56663 menggunakan .loc
sebelum_56663 = data_sebelum.loc[56663]
sesudah_56663 = data.loc[56663]

# Menyusun tabel perbandingan dengan kolom: Kolom | Skala | Sebelum | Sesudah
tabel_perbandingan_56663 = pd.DataFrame({
    "Kolom": list(data.columns),
    "Skala": [skala_ukur[col] for col in data.columns],
    "Sebelum": [sebelum_56663[col] for col in data.columns],
    "Sesudah": [sesudah_56663[col] for col in data.columns]
})

print("=== PERBANDINGAN SEBELUM & SESUDAH ENCODING (SISWA INDEKS 56663) ===")
print(tabel_perbandingan_56663.to_string(index=False))


=== PERBANDINGAN SEBELUM & SESUDAH ENCODING (SISWA INDEKS 56663) ===
                     Kolom    Skala Sebelum Sesudah
       previous_exam_score Interval   76.61   76.61
              previous_gpa Interval    3.03    3.03
     attendance_percentage    Rasio   89.84   89.84
assignment_completion_rate    Rasio   68.99   68.99
       study_hours_per_day    Rasio    2.81    2.81
          self_study_hours    Rasio    2.25    2.25
  practice_tests_completed    Rasio       2       2
     exam_preparation_days    Rasio      12      12
               sleep_hours    Rasio    6.95    6.95
              stress_level Interval      10      10
        exam_anxiety_level Interval    10.0    10.0
           exam_difficulty  Ordinal    Easy       1
             family_income  Ordinal    High       5
               pass_status  Nominal    Pass    Pass


# ==============================================================================
# PRA-PEMROSESAN DATA (DATA PREPROCESSING)
# ==============================================================================

In [9]:
# ==============================================================================
# 1. PENGECEKAN DUPLIKAT, KATEGORI UNIK, DAN TIPE DATA (DTYPES)
# ==============================================================================

# Pastikan data awal (13 fitur + target) siap untuk tahap pra-pemrosesan
data_prep = df[fitur + [target]].copy()

# a) Cek jumlah baris data duplikat
jumlah_duplikat = data_prep.duplicated().sum()
print("=" * 70)
print(f"Jumlah baris duplikat: {jumlah_duplikat}")
print("=" * 70)

# b) Kategori unik kolom kategorikal (ordinal & target nominal)
print("\n=== KATEGORI UNIK KOLOM KATEGORIKAL ===")
print(f"exam_difficulty : {data_prep['exam_difficulty'].unique().tolist()}")
print(f"family_income   : {data_prep['family_income'].unique().tolist()}")
print(f"pass_status     : {data_prep['pass_status'].unique().tolist()}")

# c) Tipe data (dtypes) seluruh 14 kolom
print("\n=== TIPE DATA (DTYPES) SETIAP KOLOM ===")
df_dtypes = pd.DataFrame({
    "Kolom": data_prep.columns,
    "Tipe Data": data_prep.dtypes.astype(str).values
})
print(df_dtypes.to_string(index=False))


Jumlah baris duplikat: 0

=== KATEGORI UNIK KOLOM KATEGORIKAL ===


exam_difficulty : ['Medium', 'Easy', 'Hard']


family_income   : ['Middle', 'High', 'Lower-Middle', 'Upper-Middle', 'Low']
pass_status     : ['Pass', 'Fail']

=== TIPE DATA (DTYPES) SETIAP KOLOM ===
                     Kolom Tipe Data
       previous_exam_score   float64
              previous_gpa   float64
     attendance_percentage   float64
assignment_completion_rate   float64
       study_hours_per_day   float64
          self_study_hours   float64
  practice_tests_completed     int64
     exam_preparation_days     int64
               sleep_hours   float64
              stress_level     int64
        exam_anxiety_level   float64
           exam_difficulty    object
             family_income    object
               pass_status    object


In [10]:
# ==============================================================================
# 2. PEMBUANGAN BARIS KOSONG (DROPNA) TANPA RESET INDEKS
# ==============================================================================

# Catat statistik sebelum dropna
baris_sebelum = len(data_prep)
na_sebelum = data_prep.isna().sum()
dist_sebelum_cnt = data_prep["pass_status"].value_counts()
dist_sebelum_pct = data_prep["pass_status"].value_counts(normalize=True) * 100

# Membuang baris yang memiliki nilai kosong tanpa reset indeks
data_clean = data_prep.dropna().copy()

# Catat statistik sesudah dropna
baris_sesudah = len(data_clean)
na_sesudah = data_clean.isna().sum()
dist_sesudah_cnt = data_clean["pass_status"].value_counts()
dist_sesudah_pct = data_clean["pass_status"].value_counts(normalize=True) * 100

# a) Perbandingan jumlah baris
print("=" * 70)
print("PERBANDINGAN JUMLAH BARIS SEBELUM & SESUDAH DROPNA")
print("=" * 70)
print(f"Baris sebelum dropna : {baris_sebelum:,}")
print(f"Baris sesudah dropna : {baris_sesudah:,}")
print(f"Baris yang dibuang   : {baris_sebelum - baris_sesudah:,} ({((baris_sebelum - baris_sesudah) / baris_sebelum) * 100:.2f}%)")

# b) Tabel nilai kosong per kolom sebelum dan sesudah
tabel_na = pd.DataFrame({
    "Nilai Kosong Sebelum": na_sebelum,
    "Nilai Kosong Sesudah": na_sesudah
})
print("\n" + "=" * 70)
print("JUMLAH NILAI KOSONG PER KOLOM (HANYA KOLOM YANG MEMILIKI MISSING)")
print("=" * 70)
print(tabel_na[tabel_na["Nilai Kosong Sebelum"] > 0])

# c) Distribusi target pass_status sebelum dan sesudah
tabel_dist_target = pd.DataFrame({
    "Jumlah (Sebelum)": dist_sebelum_cnt,
    "Persentase (%) (Sebelum)": dist_sebelum_pct.round(2),
    "Jumlah (Sesudah)": dist_sesudah_cnt,
    "Persentase (%) (Sesudah)": dist_sesudah_pct.round(2)
})
print("\n" + "=" * 70)
print("DISTRIBUSI KELAS TARGET (pass_status) SEBELUM & SESUDAH DROPNA")
print("=" * 70)
print(tabel_dist_target)


PERBANDINGAN JUMLAH BARIS SEBELUM & SESUDAH DROPNA
Baris sebelum dropna : 100,000
Baris sesudah dropna : 83,125
Baris yang dibuang   : 16,875 (16.88%)

JUMLAH NILAI KOSONG PER KOLOM (HANYA KOLOM YANG MEMILIKI MISSING)
                       Nilai Kosong Sebelum  Nilai Kosong Sesudah
previous_gpa                           7793                     0
attendance_percentage                  9863                     0

DISTRIBUSI KELAS TARGET (pass_status) SEBELUM & SESUDAH DROPNA
             Jumlah (Sebelum)  Persentase (%) (Sebelum)  Jumlah (Sesudah)  \
pass_status                                                                 
Pass                    77389                     77.39             64272   
Fail                    22611                     22.61             18853   

             Persentase (%) (Sesudah)  
pass_status                            
Pass                            77.32  
Fail                            22.68  


In [11]:
# ==============================================================================
# 3. SPLIT DATA 80:20 STRATIFIED BERDASARKAN TARGET (pass_status)
# ==============================================================================
from sklearn.model_selection import train_test_split

train_df, test_df = train_test_split(
    data_clean,
    test_size=0.20,
    random_state=42,
    stratify=data_clean["pass_status"]
)

train_df = train_df.copy()
test_df = test_df.copy()

print("=" * 70)
print("UKURAN DATASET HASIL SPLIT")
print("=" * 70)
print(f"Data Latih (Train): {train_df.shape[0]:,} baris, {train_df.shape[1]} kolom")
print(f"Data Uji   (Test) : {test_df.shape[0]:,} baris, {test_df.shape[1]} kolom")

# Distribusi kelas target pada data latih dan data uji
dist_train = pd.DataFrame({
    "Jumlah (Train)": train_df["pass_status"].value_counts(),
    "Persentase (%) (Train)": (train_df["pass_status"].value_counts(normalize=True) * 100).round(2)
})

dist_test = pd.DataFrame({
    "Jumlah (Test)": test_df["pass_status"].value_counts(),
    "Persentase (%) (Test)": (test_df["pass_status"].value_counts(normalize=True) * 100).round(2)
})

print("\n" + "=" * 70)
print("DISTRIBUSI KELAS TARGET PADA DATA LATIH & DATA UJI")
print("=" * 70)
tabel_split_dist = pd.concat([dist_train, dist_test], axis=1)
print(tabel_split_dist)


UKURAN DATASET HASIL SPLIT
Data Latih (Train): 66,500 baris, 14 kolom
Data Uji   (Test) : 16,625 baris, 14 kolom

DISTRIBUSI KELAS TARGET PADA DATA LATIH & DATA UJI
             Jumlah (Train)  Persentase (%) (Train)  Jumlah (Test)  \
pass_status                                                          
Pass                  51418                   77.32          12854   
Fail                  15082                   22.68           3771   

             Persentase (%) (Test)  
pass_status                         
Pass                         77.32  
Fail                         22.68  


In [12]:
# ==============================================================================
# 4. ANALISIS OUTLIER PER KOLOM NUMERIK MEMAKAI IQR DARI DATA LATIH
# ==============================================================================

kolom_numerik = [
    "previous_exam_score", "previous_gpa", "attendance_percentage",
    "assignment_completion_rate", "study_hours_per_day", "self_study_hours",
    "practice_tests_completed", "exam_preparation_days", "sleep_hours",
    "stress_level", "exam_anxiety_level"
]

tabel_outlier = []
iqr_bounds = {}  # Menyimpan batas IQR yang dipelajari hanya dari data latih

for col in kolom_numerik:
    q1 = train_df[col].quantile(0.25)
    q3 = train_df[col].quantile(0.75)
    iqr = q3 - q1
    batas_bawah = q1 - 1.5 * iqr
    batas_atas = q3 + 1.5 * iqr
    iqr_bounds[col] = (batas_bawah, batas_atas)
    
    jml_bawah = (train_df[col] < batas_bawah).sum()
    pct_bawah = (jml_bawah / len(train_df)) * 100
    
    jml_atas = (train_df[col] > batas_atas).sum()
    pct_atas = (jml_atas / len(train_df)) * 100
    
    skewness = train_df[col].skew()
    
    tabel_outlier.append({
        "Kolom": col,
        "Min": round(train_df[col].min(), 2),
        "Max": round(train_df[col].max(), 2),
        "Batas Bawah": round(batas_bawah, 2),
        "Batas Atas": round(batas_atas, 2),
        "Jml < BB": jml_bawah,
        "% < BB": round(pct_bawah, 2),
        "Jml > BA": jml_atas,
        "% > BA": round(pct_atas, 2),
        "Skewness": round(skewness, 2)
    })

df_analisis_outlier = pd.DataFrame(tabel_outlier)
print("=" * 105)
print("TABEL ANALISIS OUTLIER (IQR 1.5x) DAN SKEWNESS DARI DATA LATIH")
print("=" * 105)
print(df_analisis_outlier.to_string(index=False))


TABEL ANALISIS OUTLIER (IQR 1.5x) DAN SKEWNESS DARI DATA LATIH
                     Kolom   Min   Max  Batas Bawah  Batas Atas  Jml < BB  % < BB  Jml > BA  % > BA  Skewness
       previous_exam_score 30.00 100.0        38.62      100.50       213    0.32         0    0.00     -0.01
              previous_gpa  1.00   4.0         1.48        4.08       211    0.32         0    0.00     -0.02
     attendance_percentage 46.34 100.0        62.78      106.58       256    0.38         0    0.00     -0.17
assignment_completion_rate 20.00 100.0        36.86      102.54       205    0.31         0    0.00      0.02
       study_hours_per_day  0.50  12.0        -1.57        7.20         0    0.00      1742    2.62      1.13
          self_study_hours  0.25  10.0        -1.17        5.07         0    0.00      1963    2.95      1.28
  practice_tests_completed  0.00  20.0        -0.50       11.50         0    0.00      1175    1.77      0.53
     exam_preparation_days  1.00  30.0       -14.50      

In [13]:
# ==============================================================================
# 5. WINSORIZING (CLIPPING) HANYA UNTUK 3 KOLOM MEMAKAI BATAS DATA LATIH
# ==============================================================================

kolom_capping = ["study_hours_per_day", "self_study_hours", "practice_tests_completed"]
tabel_winsor = []

for col in kolom_capping:
    bb, ba = iqr_bounds[col]  # Batas IQR dari data latih
    
    # Hitung jumlah nilai yang berubah
    ubah_train = ((train_df[col] < bb) | (train_df[col] > ba)).sum()
    ubah_test = ((test_df[col] < bb) | (test_df[col] > ba)).sum()
    
    # Lakukan clipping pada data latih dan data uji
    train_df[col] = train_df[col].clip(lower=bb, upper=ba)
    test_df[col] = test_df[col].clip(lower=bb, upper=ba)
    
    tabel_winsor.append({
        "Kolom": col,
        "Batas Bawah (Train)": round(bb, 2),
        "Batas Atas (Train)": round(ba, 2),
        "Nilai Berubah (Train)": ubah_train,
        "Nilai Berubah (Test)": ubah_test
    })

df_winsor = pd.DataFrame(tabel_winsor)
print("=" * 85)
print("HASIL WINSORIZING / CLIPPING (BATAS DARI DATA LATIH)")
print("=" * 85)
print(df_winsor.to_string(index=False))


HASIL WINSORIZING / CLIPPING (BATAS DARI DATA LATIH)
                   Kolom  Batas Bawah (Train)  Batas Atas (Train)  Nilai Berubah (Train)  Nilai Berubah (Test)
     study_hours_per_day                -1.57                7.20                   1742                   406
        self_study_hours                -1.17                5.07                   1963                   499
practice_tests_completed                -0.50               11.50                   1175                   276


In [14]:
# ==============================================================================
# 6. ORDINAL ENCODING PADA DATA LATIH DAN DATA UJI
# ==============================================================================

mapping_exam_difficulty = {"Easy": 1, "Medium": 2, "Hard": 3}
mapping_family_income = {
    "Low": 1,
    "Lower-Middle": 2,
    "Middle": 3,
    "Upper-Middle": 4,
    "High": 5
}

# Terapkan pemetaan pada Train dan Test
train_df["exam_difficulty"] = train_df["exam_difficulty"].map(mapping_exam_difficulty)
test_df["exam_difficulty"] = test_df["exam_difficulty"].map(mapping_exam_difficulty)

train_df["family_income"] = train_df["family_income"].map(mapping_family_income)
test_df["family_income"] = test_df["family_income"].map(mapping_family_income)

# Assert validasi tidak ada NaN yang terbentuk
assert train_df["exam_difficulty"].isna().sum() == 0, "Ada NaN di exam_difficulty (train)!"
assert test_df["exam_difficulty"].isna().sum() == 0, "Ada NaN di exam_difficulty (test)!"
assert train_df["family_income"].isna().sum() == 0, "Ada NaN di family_income (train)!"
assert test_df["family_income"].isna().sum() == 0, "Ada NaN di family_income (test)!"

print("=" * 70)
print("ATURAN PEMETAAN ORDINAL ENCODING")
print("=" * 70)
print(f"exam_difficulty : {mapping_exam_difficulty}")
print(f"family_income   : {mapping_family_income}")
print("\nValidasi Assert: Tidak ada NaN yang terbentuk pada Train maupun Test.")


ATURAN PEMETAAN ORDINAL ENCODING
exam_difficulty : {'Easy': 1, 'Medium': 2, 'Hard': 3}
family_income   : {'Low': 1, 'Lower-Middle': 2, 'Middle': 3, 'Upper-Middle': 4, 'High': 5}

Validasi Assert: Tidak ada NaN yang terbentuk pada Train maupun Test.


In [15]:
# ==============================================================================
# 7. OVERSAMPLING KELAS MINORITAS (DATA LATIH SAJA) DENGAN sklearn.utils.resample
# ==============================================================================
from sklearn.utils import resample

dist_train_sebelum = train_df["pass_status"].value_counts()

# Pisahkan kelas mayoritas (Pass) dan minoritas (Fail) pada data latih
train_majority = train_df[train_df["pass_status"] == "Pass"]
train_minority = train_df[train_df["pass_status"] == "Fail"]

# Lakukan oversampling pada kelas minoritas menyamai jumlah kelas mayoritas
minority_upsampled = resample(
    train_minority,
    replace=True,                  # Sampel dengan pengembalian
    n_samples=len(train_majority), # Samakan jumlah dengan kelas Pass
    random_state=42
)

# Gabungkan kembali dan acak urutan barisnya (tanpa reset indeks)
train_resampled = pd.concat([train_majority, minority_upsampled])
train_resampled = train_resampled.sample(frac=1, random_state=42)

# Distribusi sesudah oversampling
dist_train_sesudah = train_resampled["pass_status"].value_counts()

tabel_resample = pd.DataFrame({
    "Jumlah (Sebelum)": dist_train_sebelum,
    "Persentase (%) (Sebelum)": (dist_train_sebelum / len(train_df) * 100).round(2),
    "Jumlah (Sesudah)": dist_train_sesudah,
    "Persentase (%) (Sesudah)": (dist_train_sesudah / len(train_resampled) * 100).round(2)
})

print("=" * 70)
print("DISTRIBUSI KELAS TARGET DATA LATIH SEBELUM & SESUDAH OVERSAMPLING")
print("=" * 70)
print(tabel_resample)
print(f"\nCatatan: Data uji (test_df) tetap orisinal ({len(test_df):,} baris).")


DISTRIBUSI KELAS TARGET DATA LATIH SEBELUM & SESUDAH OVERSAMPLING
             Jumlah (Sebelum)  Persentase (%) (Sebelum)  Jumlah (Sesudah)  \
pass_status                                                                 
Pass                    51418                     77.32             51418   
Fail                    15082                     22.68             51418   

             Persentase (%) (Sesudah)  
pass_status                            
Pass                             50.0  
Fail                             50.0  

Catatan: Data uji (test_df) tetap orisinal (16,625 baris).


In [16]:
# ==============================================================================
# 8. PEMISAHAN X_train, y_train, X_test, y_test DAN SAMPEL HASIL AKHIR
# ==============================================================================

# Pemisahan fitur prediktor (X) dan target prediksi (y)
X_train = train_resampled.drop(columns=["pass_status"])
y_train = train_resampled["pass_status"]

X_test = test_df.drop(columns=["pass_status"])
y_test = test_df["pass_status"]

print("=" * 70)
print("DIMENSI DATA AKHIR (SHAPE)")
print("=" * 70)
print(f"X_train shape: {X_train.shape}")
print(f"y_train shape: {y_train.shape}")
print(f"X_test shape : {X_test.shape}")
print(f"y_test shape : {y_test.shape}")

print("\n" + "=" * 70)
print("5 BARIS TERATAS DATA LATIH HASIL PEMROSESAN AKHIR (FITUR + TARGET)")
print("=" * 70)
print(train_resampled.head())

print("\n" + "=" * 70)
print("5 BARIS TERBAWAH DATA LATIH HASIL PEMROSESAN AKHIR (FITUR + TARGET)")
print("=" * 70)
print(train_resampled.tail())


DIMENSI DATA AKHIR (SHAPE)
X_train shape: (102836, 13)
y_train shape: (102836,)
X_test shape : (16625, 13)
y_test shape : (16625,)

5 BARIS TERATAS DATA LATIH HASIL PEMROSESAN AKHIR (FITUR + TARGET)


       previous_exam_score  previous_gpa  attendance_percentage  \
65155                76.54          3.21                  84.21   
71522                67.70          2.86                  74.99   
84297                55.58          2.28                  91.08   
5865                 47.58          1.87                  78.45   
84902                84.76          3.34                  79.27   

       assignment_completion_rate  study_hours_per_day  self_study_hours  \
65155                       74.61                 1.80              1.22   
71522                       60.38                 0.69              0.57   
84297                       65.58                 2.95              1.80   
5865                        67.99                 1.11              0.73   
84902                       71.37                 4.33              3.85   

       practice_tests_completed  exam_preparation_days  sleep_hours  \
65155                       8.0                     13         6.13  